# Amazon ML Challenge 2026 — Phase 2B lexical retrieval

Thin launcher for bounded name/address word TF-IDF plus transliteration, rare-token, and numeric rescue. Attach the raw data, Phase-1 normalized output, and Phase-2A exact/structured output. CPU only; GPU is not used. **UNVERIFIED estimate:** 2–8 hours. Monitor working-disk capacity because K=15 artifacts are large.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
REPO_URL='https://github.com/yugtheguy/NEW_AMAZON.git'
PINNED_COMMIT='6cfd0425c709ff88c601715b5541dd230004c118'
INPUT_ROOT=Path('/kaggle/input')
REPO_DIR=Path('/kaggle/working/NEW_AMAZON')
RESUME_ARTIFACTS_ROOT=None
print({'repo':REPO_URL,'commit':PINNED_COMMIT})

In [ ]:
expected={'train_source1.tsv','train_source2.tsv','train_source3.tsv','train_ground_truth.tsv','test_source1.tsv','test_source2.tsv','test_source3.tsv'}
found={name:list(INPUT_ROOT.rglob(name)) for name in sorted(expected)}
problems={name:paths for name,paths in found.items() if len(paths)!=1}
if problems: raise RuntimeError(f'Raw inputs must occur exactly once: {problems}')
normalization_manifests=[]; exact_manifests=[]
for path in INPUT_ROOT.rglob('part-*.manifest.json'):
    try: stage=json.loads(path.read_text(encoding='utf-8')).get('stage')
    except Exception: continue
    if stage=='normalization': normalization_manifests.append(path)
    elif stage=='exact_structured': exact_manifests.append(path)
normalized_roots=sorted({p.parents[3] for p in normalization_manifests})
exact_roots=sorted({p.parents[3] for p in exact_manifests})
if len(normalized_roots)!=1: raise RuntimeError(f'Expected one Phase-1 normalized root: {normalized_roots}')
if len(exact_roots)!=1: raise RuntimeError(f'Expected one Phase-2A exact_structured root: {exact_roots}')
NORMALIZED_ROOT=normalized_roots[0]; EXACT_STRUCTURED_ROOT=exact_roots[0]
print({'normalized_root':str(NORMALIZED_ROOT),'normalization_manifests':len(normalization_manifests)})
print({'exact_structured_root':str(EXACT_STRUCTURED_ROOT),'exact_manifests':len(exact_manifests)})

In [ ]:
if not REPO_DIR.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(REPO_DIR)],check=True)
else:
    status=subprocess.run(['git','-C',str(REPO_DIR),'status','--porcelain'],capture_output=True,text=True,check=True).stdout
    if status.strip(): raise RuntimeError(f'Existing repository is not clean: {status}')
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',PINNED_COMMIT],check=True)
actual=subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
assert actual==PINNED_COMMIT,(actual,PINNED_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[data,retrieval]','duckdb==1.5.5'],check=True)
print('Installed',actual)

In [ ]:
artifact_root=REPO_DIR/'artifacts'
if RESUME_ARTIFACTS_ROOT is not None:
    resume=Path(RESUME_ARTIFACTS_ROOT)
    for relative in [Path('retrieval/v1')/name for name in ('name_word','address_word','transliteration','rare_token','numeric')]:
        source=resume/relative; destination=artifact_root/relative
        if source.exists(): shutil.copytree(source,destination,dirs_exist_ok=True)
subprocess.run(['bash','-lc','free -h && df -h /kaggle/working'],check=True)

In [ ]:
log_path=Path('/kaggle/working/phase2b_lexical.log')
command=[sys.executable,'-m','amazon_er.cli','run','--stage','retrieve-lexical','--config',str(REPO_DIR/'configs/prod.yaml'),'--normalized-root',str(NORMALIZED_ROOT),'--exact-structured-root',str(EXACT_STRUCTURED_ROOT),'--data-root',str(INPUT_ROOT)]
print('Running:',' '.join(command))
with log_path.open('a',encoding='utf-8') as log:
    process=subprocess.Popen(command,cwd=REPO_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
    code=process.wait()
if code!=0: raise RuntimeError(f'Phase 2B failed with exit code {code}; inspect {log_path}')
print('Phase 2B complete')

In [ ]:
report=REPO_DIR/'artifacts/reports/retrieval/v1/lexical/audit.json'
audit=json.loads(report.read_text(encoding='utf-8'))
assert [item['k'] for item in audit['k_sweep']]==[3,5,8,10,15]
for family in ('name_word','address_word','transliteration','rare_token','numeric'):
    root=REPO_DIR/'artifacts/retrieval/v1'/family
    parts=list(root.rglob('part-*.parquet')); manifests=list(root.rglob('part-*.manifest.json'))
    assert parts and len(parts)==len(manifests),(family,len(parts),len(manifests))
    print(family,len(parts))
print(json.dumps(audit,indent=2,sort_keys=True))
print('Use Save Version to preserve artifacts and',log_path)